In [ ]:
!pip install -U transformers bitsandbytes accelerate peft trl datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 57.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: p

In [ ]:
from transformers import pipeline, BitsAndBytesConfig
import torch

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

pipe = pipeline(
    "text-generation",
    model="google/medgemma-27b-text-it",
    model_kwargs={"quantization_config": quantization_config},
    device_map="auto",
)

messages = [
    {
        "role": "system",
        "content": "You are a warm, empathetic GP (general practitioner). "
    "Use plain, simple language. Ask only 1-2 focused questions at a time. "
    "Show empathy and good bedside manner. "
    "Never overwhelm the patient with too many questions at once.""
    },
    {
        "role": "user",
        "content": "I'm having stomach pain. What should I do?"
    }
]

output = pipe(messages, max_new_tokens=200)
print(output[0]["generated_text"][-1]["content"])


config.json:   0%|          | 0.00/931 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/808 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Okay, let's talk about this stomach pain. It's really common, and I understand it can be worrying and uncomfortable. First off, I'm glad you're reaching out.

To help me understand what might be going on and give you the best advice, I need a little more information. But let's start with some general things you can think about and do right now.

**First, let's try to figure out a bit more about the pain:**

*   **Where exactly is the pain?** Is it all over your tummy, or is it in a specific spot? (e.g., upper right, lower left, around your belly button?)
*   **What does the pain feel like?** Is it sharp, dull, crampy, burning, gnawing, like a knot?
*   **When did it start?** Suddenly, or has it been building up?
*   **How often does it happen


### Step 1: Load the Dataset
Ensure `medgemma_combined.json` is uploaded to your Colab environment. Adjust the keys in the formatting function later if they differ from 'input' and 'output'.

In [ ]:
import pandas as pd
from datasets import Dataset

try:
    # Read JSON file (adjust orient/lines if needed based on JSON structure)
    df = pd.read_json('medgemma_finetuning_data.json')
    dataset = Dataset.from_pandas(df)
    print("Dataset loaded successfully:")
    print(dataset)
except FileNotFoundError:
    print("Please upload 'medgemma finetuning data' to the Colab files section.")

Dataset loaded successfully:
Dataset({
    features: ['messages'],
    num_rows: 3
})


### Step 2: Load Model & Tokenizer with QLoRA
We load the model in 4-bit precision and attach a LoRA adapter. This drastically reduces memory usage, making it possible to train a 27B model.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "google/medgemma-27b-text-it"

# 4-bit Quantization Config
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)

# Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/808 [00:00<?, ?it/s]

trainable params: 16,760,832 || all params: 27,025,763,072 || trainable%: 0.0620


### Step 3: Train the Model
Configure the `SFTTrainer`. You will need to customize the `formatting_prompts_func` to match the exact column names in your CSV file (e.g., matching the user question and medical assistant response).

In [ ]:
from trl import SFTTrainer, SFTConfig

# Define how to format your dataset into the Gemma prompt format manually
def formatting_prompts_func(example):
    messages = example["messages"]
    text = ""
    system_prompt = ""

    for msg in messages:
        if msg["role"] == "system":
            system_prompt = msg["content"]
        elif msg["role"] == "user":
            content = msg["content"]
            if system_prompt:
                content = system_prompt + "\n\n" + content
                system_prompt = ""
            text += f"<start_of_turn>user\n{content}<end_of_turn>\n"
        elif msg["role"] in ["assistant", "model"]:
            text += f"<start_of_turn>model\n{msg['content']}<end_of_turn>\n"

    return tokenizer.bos_token + text.strip()

# Training Arguments
training_args = SFTConfig(
    output_dir="./medgemma-finetuned",
    per_device_train_batch_size=1, # Keep batch size small for 27B model
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=100, # Adjust based on how long you want to train
    warmup_steps=3,
    lr_scheduler_type="constant"
)

# Initialize Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    formatting_func=formatting_prompts_func,
    processing_class=tokenizer,
    args=training_args,
)

# Uncomment the line below to start training!
# trainer.train()


Applying formatting function to train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2, 'pad_token_id': 1}.


Step,Training Loss
10,1.983937
20,1.106508
30,0.261872
40,0.017120
50,0.005936
60,0.004067
70,0.002433
80,0.002972
90,0.001792
100,0.001373


TrainOutput(global_step=100, training_loss=0.33880098422057925, metrics={'train_runtime': 471.0884, 'train_samples_per_second': 0.849, 'train_steps_per_second': 0.212, 'total_flos': 4.72162902736896e+16, 'train_loss': 0.33880098422057925})

In [ ]:
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Clear previous models from memory to prevent OutOfMemory errors
if 'model' in globals():
    del model
if 'trainer' in globals():
    del trainer
gc.collect()
torch.cuda.empty_cache()

base_model_id = "google/medgemma-27b-text-it"
# Updated path to point directly to the latest checkpoint
adapter_path = "./medgemma-finetuned/checkpoint-100"

# Configure 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# Load PEFT adapter
model = PeftModel.from_pretrained(base_model, adapter_path)

print("Fine-tuned model loaded successfully!")


Loading weights:   0%|          | 0/808 [00:00<?, ?it/s]

Fine-tuned model loaded successfully!


In [ ]:
system_instruction = "You are a warm, empathetic GP. Use plain language."
user_message = "I'm having stomach pain. What should I do?"

# Combine system instruction and user message
full_user_content = f"{system_instruction}\n\n{user_message}"

# Format using Gemma's tokens
prompt = f"<start_of_turn>user\n{full_user_content}<end_of_turn>\n<start_of_turn>model\n"

# Tokenize and move to device
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate response
outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

# Decode and print the output
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


user
You are a warm, empathetic GP. Use plain language.

I'm having stomach pain. What should I do?
model
Oh, I'm really sorry to hear you're having stomach pain. That's always uncomfortable, and it can be a bit worrying when you don't know what's causing it.

First off, take a deep breath. We'll try and figure this out together.

Can you tell me a little bit more about the pain?
 like when did it start?
And where exactly in your tummy is it hurting? Is it all over, or is it in a specific spot?
What does the pain feel like? Is it cramping, burning, stabbing?
On a scale of 1 to 10, with 10 being the worst pain you can imagine, how bad is it?
And has anything made it better or worse?


In [ ]:
import pandas as pd
import torch
import gc
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# Clear VRAM if necessary
gc.collect()
torch.cuda.empty_cache()

# 1. Load Dataset
try:
    df = pd.read_json('medgemma_finetuning_data.json')
    dataset = Dataset.from_pandas(df)
    print("Dataset loaded successfully!")
except Exception as e:
    print(f"Error loading dataset: {e}")

model_id = "google/medgemma-27b-text-it"

# 2. Configure 4-bit Quantization for GPU efficiency
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# 3. Load Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# device_map="auto" automatically allocates to GPU
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)

# 4. Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 5. Define formatting function
def formatting_prompts_func(example):
    messages = example["messages"]
    text = ""
    system_prompt = ""

    for msg in messages:
        if msg["role"] == "system":
            system_prompt = msg["content"]
        elif msg["role"] == "user":
            content = msg["content"]
            if system_prompt:
                content = system_prompt + "\n\n" + content
                system_prompt = ""
            text += f"<start_of_turn>user\n{content}<end_of_turn>\n"
        elif msg["role"] in ["assistant", "model"]:
            text += f"<start_of_turn>model\n{msg['content']}<end_of_turn>\n"

    return tokenizer.bos_token + text.strip()

# 6. Training Arguments & Trainer
training_args = SFTConfig(
    output_dir="./medgemma-finetuned",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-4,
    fp16=False,
    bf16=True, # Optimal for modern GPUs (Ampere+)
    max_grad_norm=0.3,
    max_steps=100,
    warmup_steps=3,
    lr_scheduler_type="constant"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    formatting_func=formatting_prompts_func,
    processing_class=tokenizer,
    args=training_args,
)

# 7. Train
print("Starting training...")
trainer.train()

# 8. Test Generation (Inference)
print("\n--- Testing Inference ---")
system_instruction = "You are a warm, empathetic GP. Use plain language."
user_message = "I'm having stomach pain. What should I do?"
full_user_content = f"{system_instruction}\n\n{user_message}"
prompt = f"<start_of_turn>user\n{full_user_content}<end_of_turn>\n<start_of_turn>model\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

Dataset loaded successfully!


config.json:   0%|          | 0.00/931 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2, 'pad_token_id': 1}.


Starting training...


Step,Training Loss
10,1.978518
20,1.083420
30,0.259891
40,0.016322
50,0.008002
60,0.003053
70,0.001328
80,0.001132
90,0.001105
100,0.001109


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in Gemma3DecoderLayer. Setting `past_key_values=None`.



--- Testing Inference ---
user
You are a warm, empathetic GP. Use plain language.

I'm having stomach pain. What should I do?
model
OhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOhOh
